# 05 — Evaluate fresh, matched runs

Score the same frozen cell identities in the same cores; missing predicted cells remain zero. A failed run, incomplete launcher, changed seed or output gene excess blocks a clean comparison. All native counts remain unchanged. The strict fresh-run mode has no historical recovery exceptions.

In [1]:
from pathlib import Path
import os, sys, json
import pandas as pd
from IPython.display import display
SUITE_DIR = next((p for parent in [Path.cwd(), *Path.cwd().parents]
                  for p in [parent, parent / "suite"] if (p / "benchmark_session.py").is_file()), None)
if SUITE_DIR is None:
    raise FileNotFoundError("Open this notebook from the unpacked public benchmark (or its suite/ directory).")
sys.path.insert(0, str(SUITE_DIR))
from benchmark_session import start
from p06212_benchmark.common import load_config, output_root, write_json, read_json
CONFIG_PATH, config, ROOT = start()

ACTIVE SPECIMEN: ovarian 
Configuration: /home/domino/reny28/Projects/ShortProject/Xenium_pseudo_VisiumHD01/public_prime_benchmark/ovarian/pilot_01/config.json 
Run directory: /home/domino/reny28/Projects/ShortProject/Xenium_pseudo_VisiumHD01/public_prime_benchmark/ovarian/pilot_01/results_root


In [2]:
from datetime import datetime, timezone
import numpy as np
from p06212_benchmark.evaluation import evaluate_h5ad
from p06212_benchmark.evaluation_report import make_report
from p06212_benchmark.common import file_sha256
from p06212_benchmark.fresh_audit import audit_selected
manifest = pd.read_csv(ROOT / 'condition_manifest.csv', dtype={'bundle_id':str,'condition_id':str})
lookup = manifest.set_index(['bundle_id','condition_id'], verify_integrity=True)
SCOPE = "pilot"  # switch to 'all' after all bundles have fresh completed runs
REPEAT_IDS = [1]
methods = ['bin2cell','smurf_'+config['methods']['smurf']['variant'],'proseg']
base_ids = list(dict.fromkeys(manifest.loc[manifest.condition_id.eq("baseline"),"bundle_id"]))
if not base_ids or SCOPE not in ('pilot','all'): raise ValueError("Unknown scope or no baseline bundles.")
bundle_ids = base_ids[:1] if SCOPE=="pilot" else base_ids
REPORT = ROOT/'reports'/('fresh_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
REPORT.mkdir(parents=True, exist_ok=False)
audit, result_files = audit_selected(ROOT, lookup, bundle_ids, methods, REPEAT_IDS)
audit.to_csv(REPORT/'fresh_run_audit.csv',index=False)
display(audit)
if audit.empty or not audit.status.eq('verified').all():
    raise ValueError(f'Comparison paused. Inspect {REPORT}/fresh_run_audit.csv: missing, failed, incomplete, mismatched or excess-count runs are not silently dropped.')

,bundle_id,method,repeat,condition,status,reason,input_counts,output_counts,gene_excess_counts,n_excess_genes,seed_sha256,total_runner_seconds
0,ovarian_B01,bin2cell,1,baseline,verified,,4871608,4439808,0,0,59d428c108fe9d5b9e39823e7edff58634e3fa0712fd5a...,27.038884
1,ovarian_B01,smurf_lite,1,baseline,unverified,ValueError: Native count excess at the bundle/...,4871608,4073500,3,3,59d428c108fe9d5b9e39823e7edff58634e3fa0712fd5a...,1090.567405
2,ovarian_B01,proseg,1,baseline,verified,,4871608,4753511,0,0,59d428c108fe9d5b9e39823e7edff58634e3fa0712fd5a...,301.511966


ValueError: Comparison paused. Inspect /home/domino/reny28/Projects/ShortProject/Xenium_pseudo_VisiumHD01/public_prime_benchmark/ovarian/pilot_01/results_root/reports/fresh_20261123T215805_933895Z/fresh_run_audit.csv: missing, failed, incomplete, mismatched or excess-count runs are not silently dropped.

### Compute raw count agreement
Xenium IDs supply a reference standard, not an independent biological truth. Working counts stay in evaluator-only directories.

In [ ]:
failures=[]
for prediction_path in result_files:
    bundle, condition, method, repeat_name = prediction_path.relative_to(ROOT / "results").parts[:4]
    row = lookup.loc[(bundle,condition)]
    reference_dir = Path(row.reference_dir)
    mapping_review = reference_dir / "mapping_review.json"
    if not mapping_review.exists():
        raise FileNotFoundError(f"Review/freeze the baseline seed map first: {reference_dir}")
    proof = read_json(mapping_review)
    if not proof.get("frozen") or file_sha256(reference_dir / "seed_map.csv") != proof["seed_map_sha256"]:
        raise ValueError("The frozen identity map changed after review.")
    input_meta = read_json(Path(row.input_dir) / "metadata.json")
    baseline_hash = input_meta.get("baseline_seeds_sha256", input_meta.get("seeds_sha256"))
    if baseline_hash != proof["baseline_seeds_sha256"]:
        raise ValueError("These method inputs do not originate from the reviewed baseline mask.")
    frozen = pd.read_csv(reference_dir / "seed_map.csv",
        dtype={"seed_id":str,"reference_cell_id":str,"sample_id":str,"component_id":str,"core_id":str})
    frozen["bundle_id"] = bundle
    frozen["dx_um"] = float(row.dx_um); frozen["dy_um"] = float(row.dy_um)
    frozen["added_offset_um"] = float(np.hypot(row.dx_um,row.dy_um))
    destination = REPORT / "evaluation" / bundle / condition / method / repeat_name
    try:
        result = evaluate_h5ad(prediction_path, reference_dir / "counts.h5ad", frozen,
                               method=method, condition=condition, replicate=int(repeat_name.split("_")[-1]))
        result.save(destination)
        print(bundle,condition,method,repeat_name,"scored:",len(result.cell_metrics))
    except Exception as exc:
        failures.append({"prediction":str(prediction_path),"error":f"{type(exc).__name__}: {exc}"})
write_json(REPORT / "evaluation_failures.json", failures)
if failures:
    display(pd.DataFrame(failures))
    raise RuntimeError("Resolve evaluation failures before interpreting a method comparison.")

### Write per-core, cell-type, coverage and timing tables

In [ ]:
from p06212_benchmark.evaluation_report import collect_evaluations, plot_benchmark_report
tables = collect_evaluations(REPORT/'evaluation', ROOT/'results')
wanted = set(str(Path(p).parent/'run_summary.json') for p in result_files)
tables.runs = tables.runs.loc[tables.runs.source_file.isin(wanted)].copy()
if set(tables.runs.source_file) != wanted:
    raise ValueError("Selected timing records do not match scored fresh runs.")
expected = set(pd.read_csv(ROOT/'review'/'selected_cores.csv',dtype=str).query('bundle_id in @bundle_ids').core_id)
for (m,rep), group in tables.cores.groupby(['method','replicate'],observed=True):
    if set(group.core_id.astype(str)) != expected:
        raise ValueError(f'{m} repeat {rep}: expected matching held-out core coverage.')
for filename, frame in [('all_core_metrics',tables.cores),('all_celltype_metrics',tables.celltypes),
                        ('all_reference_seed_coverage',tables.coverage),('all_runtime_audits',tables.runs),
                        ('all_evaluation_audits',tables.audits)]:
    frame.to_csv(REPORT/(filename+'.csv'),index=False)
plot_benchmark_report(tables, REPORT/'benchmark_summary.png', title=f"{next(iter(config['samples']))} · reference agreement")
from IPython.display import Image
display(Image(filename=str(REPORT/'benchmark_summary.png')))
display(tables.cores[['core_id','method','normalized_l1_gene_count_error','median_profile_cosine','gene_count_recovery']])
print('Fresh comparable report:', REPORT)

Summary is descriptive. Report detector coverage, L1, counts, cosine and lineage evidence together. Multiple cores within one specimen are not independent donors. Compare methods within specimen first.